In [15]:
import pandas as pd
import numpy as np

In [16]:
chains = pd.read_csv(r"C:\Users\Junayed\defillama-evm-chain-analysis\data\raw\chains_list.csv")

daily_tvl = pd.read_csv(r"C:\Users\Junayed\defillama-evm-chain-analysis\data\raw\chain_tvl_daily.csv")

print(chains.shape, daily_tvl.shape)
print(daily_tvl["chain"].nunique(), "chains available")
print(daily_tvl.head())
print(daily_tvl.dtypes)

(467, 7) (499657, 3)
467 chains available
      chain        date     tvl
0  Moonbeam  1626220800  395004
1  Moonbeam  1626307200  392102
2  Moonbeam  1626393600  391047
3  Moonbeam  1626480000  386780
4  Moonbeam  1626566400  388468
chain      str
date     int64
tvl      int64
dtype: object


In [17]:
chains.info()

<class 'pandas.DataFrame'>
RangeIndex: 467 entries, 0 to 466
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   gecko_id         315 non-null    str    
 1   gasTokenGeckoId  368 non-null    str    
 2   tvl              467 non-null    float64
 3   tokenSymbol      397 non-null    str    
 4   cmcId            309 non-null    float64
 5   name             467 non-null    str    
 6   chainId          250 non-null    float64
dtypes: float64(3), str(4)
memory usage: 36.1 KB


In [ ]:
daily_tvl.info()

<class 'pandas.DataFrame'>
RangeIndex: 499657 entries, 0 to 499656
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   chain   499657 non-null  str  
 1   date    499657 non-null  int64
 2   tvl     499657 non-null  int64
dtypes: int64(2), str(1)
memory usage: 14.7 MB


In [ ]:
daily_tvl.dtypes

chain      str
date     int64
tvl      int64
dtype: object

### Data Scoping & Temporal Standardization

To align the dataset with the core research question, we scope the daily Total Value Locked (TVL) series strictly to the two-year evaluation window: **October 1, 2024 to September 30, 2026**. Older observations dating back to September 2017 are excluded to eliminate historical regime noise.

* **Timestamp Conversion:** The UNIX epoch values in `date` were converted to standard pandas datetime objects (`pd.to_datetime(..., unit="s")`).
* **Boundary Verification:** 
  * Earliest recorded date: `2017-09-27 00:00:00`
  * Latest recorded date: `2026-09-30 00:00:00`
* **Midnight Alignment (`all midnight: True`):** Validating `(daily_tvl["date"] == daily_tvl["date"].dt.normalize()).all()` confirms that every entry sits precisely at midnight (`00:00:00`). This ensures clean boundary filtering and consistent 24-hour spacing without intraday timestamp drift.
* **Scoping Execution:** Records prior to `2024-10-01` are truncated, isolating the target 24-month horizon into an independent DataFrame workspace for downstream modeling and feature generation.

In [ ]:
daily_tvl["date"] = pd.to_datetime(daily_tvl["date"], unit="s")

print(daily_tvl["date"].min(), "->", daily_tvl["date"].max())

print("all midnight:", (daily_tvl["date"] == daily_tvl["date"].dt.normalize()).all())

2017-09-27 00:00:00 -> 2026-09-30 00:00:00
all midnight: True


### Window Isolation & Chain Coverage

To implement the two-year project scope, the daily Total Value Locked (TVL) series was filtered between **October 1, 2024** and **September 30, 2026** using explicit boundary masking:

* **Temporal Window:** `window_start = "2024-10-01"` to `window_end = "2026-09-30"`
* **Records Retained:** **292,308** daily observations remain after truncating out-of-scope historical records.
* **Network Diversity:** **467** unique chains (`daily_tvl["chain"].nunique()`) are active within this 24-month horizon.
* **Workspace Safety:** Applying `.copy()` severs references to the original DataFrame, guaranteeing that downstream metric computations, aggregations, and chain-level analyses proceed without triggering SettingWithCopy warnings.

In [ ]:
window_start = "2024-10-01"
window_end = "2026-09-30"

in_window = (daily_tvl["date"] >= window_start) & (daily_tvl["date"] <= window_end)

daily_tvl = daily_tvl[in_window].copy()

print(f"row kept: {len(daily_tvl):,} | chains within the window: {daily_tvl["chain"].nunique()}")

row kept: 292,308 | chains within the window: 467


In [20]:
chains["chainId"].isna().sum()

np.int64(217)